# T4 — DVC: versionamento de dados

**Objetivo:** versionar dados como versionamos código, sem colocar os dados no Git, e ligar cada modelo
à versão exata dos dados que o treinou.

| Peça | Onde fica | Vai para o Git? |
|---|---|---|
| `bike.csv` (o dado) | sua pasta + cache `.dvc/cache` + **remote** | ❌ |
| `bike.csv.dvc` (ponteiro: hash md5 + tamanho) | ao lado do dado | ✅ |
| `.dvc/config` (onde fica o remote) | `.dvc/` | ✅ |

> 💡 O DVC e o remote (`../dvc-storage`) foram configurados na T1. Lá, o pipeline só **anotava o hash** do
> `bike.csv` no `dvc.lock`. Agora o próprio dado passa a ser versionado.
>
> Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`), para não rodarem sem querer.
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

---
# Parte 1 · Juntos

In [2]:
%%bash
git status --short
git log --oneline | head -5
dvc remote list

?? aula01/_run_t0_ambiente.ipynb
?? aula01/_run_t1_pipeline_dvc.ipynb
?? aula01/_run_t2_mlflow_track

ing.ipynb
?? aula01/_run_t3_model_registry.ipynb
?? aula01/_run_t4_dvc_dados.ipynb


8b071f5 HGB: max_leaf_nodes 15
377d758 Pipeline DVC: + evaluate e plot
93d510a Pipeline DVC: prepare

 -> split -> train
b9786f0 Inicializa DVC com remote local
c849714 Template inicial


local   /workspaces/dvc-storage  (default)


## 1. Versionar os dados v1 (2011)

São sempre os mesmos 4 passos. **Guarde esta receita: você vai repeti-la na Parte 2.**

| Passo | Comando | O que faz |
|---|---|---|
| 1 | `dvc add data/raw/bike.csv` | calcula o md5, guarda no cache e cria o ponteiro `.dvc` |
| 2 | `git add data/raw/bike.csv.dvc` + `git commit` | o **ponteiro** vai para o Git |
| 3 | `git tag dados-v1` | um nome fácil para esta versão |
| 4 | `dvc push` | o **dado** vai para o remote |

In [3]:
%%bash
# dvc add data/raw/bike.csv
cat data/raw/bike.csv.dvc

⠋ Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc



To enable auto staging, run:

	dvc config core.autostage true


outs:
- md5: fc7d423afbce5699a61ac1a8acc533b8
  size: 562742
  hash: md5
  path: bike.csv


O `.gitignore` do template já ignora CSVs dentro de `data/`, por isso o DVC não precisou criar um novo.

In [4]:
%%bash
# git add data/raw/bike.csv.dvc
# git commit -m "Dados v1: 2011"
# git tag dados-v1
# dvc push

[main 61d37b6] Dados v1: 2011
 1 file changed, 5 insertions(+)
 create mode 100644 data/raw/bike.csv

.dvc


1 file pushed

## 2. Desastre: apagaram o dado **e** o cache

Se o dado está no remote, basta um `dvc pull`.

In [5]:
%%bash
# rm -rf data/raw/bike.csv .dvc/cache
# dvc pull
wc -l data/raw/bike.csv

A       data/raw/bike.csv


9 files fetched and 1 file added

8646 data/raw/bike.csv


## 3. Como voltar no tempo (você vai usar na Parte 2)

Para trazer uma versão antiga são **2 comandos**: o Git traz o **ponteiro** antigo e o
`dvc checkout` traz o **dado** correspondente.

```bash
git checkout <tag> -- data/raw/bike.csv.dvc   # ponteiro da versão <tag>
dvc checkout                                  # dado que corresponde ao ponteiro
```
Para voltar à versão atual, troque `<tag>` por `HEAD`.

---
# Parte 2 · Sozinhos

Chegaram dados novos: o **1º semestre de 2012**. Versione-os com a mesma receita da Parte 1.
Repare no `dvc status`: ele mostra que o dado mudou **e** que o estágio `prepare` do pipeline, que depende dele, ficou desatualizado.

In [6]:
%%bash
# python src/get_data.py --until 2012-06-30
dvc status

data/raw/bike.csv: 13,003 linhas | 2011-01-01 a 2012-06-30


prepare:
	changed deps:
		modified:           data/raw/bike.csv


data/raw/bike.csv.dvc:


	changed outs:
		modified:           data/raw/bike.csv


### 1. Versione os dados v2 (receita do passo 1)

In [7]:
%%bash
# dvc add data/raw/bike.csv
# git add data/raw/bike.csv.dvc
# git commit -m "Dados v2: até jun/2012"
# git tag dados-v2
# dvc push
git log --oneline --decorate | head -5

⠋ Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc



To enable auto staging, run:

	dvc config core.autostage true


[main 3728563] Dados v2: até jun/2012
 1 file changed, 2 insertions(+), 2 deletions(-)


1 file pushed


3728563 (HEAD -> main, tag: dados-v2) Dados v2: até jun/2012
61d37b6 (tag: dados-v1) Dados v1: 2011


8b071f5 (origin/main) HGB: max_leaf_nodes 15
377d758 Pipeline DVC: + evaluate e plot
93d510a Pipeli

ne DVC: prepare -> split -> train


### 2. Viagem no tempo: volte para a v1, conte as linhas e retorne para a v2 (passo 3)

In [8]:
%%bash
# git checkout dados-v1 -- data/raw/bike.csv.dvc
# dvc checkout
wc -l data/raw/bike.csv      # v1: 8.646 linhas (8.645 + cabeçalho)

M       data/raw/bike.csv


8646 data/raw/bike.csv


In [9]:
%%bash
# git checkout HEAD -- data/raw/bike.csv.dvc
# dvc checkout
wc -l data/raw/bike.csv      # v2: 13.004 linhas

M       data/raw/bike.csv


13004 data/raw/bike.csv


### 3. Retreine o pipeline com os dados v2 (o `dvc repro` da T1)

Como o dado mudou, o pipeline inteiro roda de novo. O treino gera um novo run no MLflow e uma nova versão no Registry.

In [10]:
%%bash
# dvc repro
dvc metrics show

'data/raw/bike.csv.dvc' didn't change, skipping


Running stage 'prepare':


> python src/prepare.py


prepare: 13,003 linhas (0 removidas) e 14 colunas -> data/prepared/bike.csv


Updating lock file 'dvc.lock'


Running stage 'split':


... (25 outputs left out)


To track the changes with git, run:

	git add dvc.lock



To enable auto staging, run:

	dvc config core.autostage true


Use `dvc push` to send your updates to remote storage.


Path          mae      r2       rmse
metrics.json  35.5516  0.94818  50.53559


### 4. Guarde o recibo e publique no GitHub: o código, os ponteiros e as tags (os dados não vão)

In [11]:
%%bash
# git add dvc.lock metrics.json reports/previsto_vs_real.png
# git commit -m "Pipeline retreinado com os dados v2"
# dvc push
# git push
# git push --tags

[main 1a7773d] Pipeline retreinado com os dados v2
 3 files changed, 27 insertions(+), 27 deletions(

-)


8 files pushed

To https://github.com/aluno/mlops-facens.git
   8b071f5..1a7773d  main -> main


To https://github.com/aluno/mlops-facens.git
 * [new tag]         dados-v1 -> dados-v1
 * [new tag]         dado

s-v2 -> dados-v2


---
## Para discutir
1. Qual é o md5 do `bike.csv` em cada tag? Repare que conseguimos ver **sem fazer checkout**.
2. Por que o `bike.csv` nunca aparece no `git status`?
3. **Linhagem:** qual versão dos dados treinou o modelo mais recente do pipeline? Compare a tag `data_version` do run
   (MLflow → experimento `bike-demand-pipeline`) com os md5 abaixo. O que essa tag mostrava nos runs da T1?

In [12]:
%%bash
for tag in dados-v1 dados-v2; do
  echo "== $tag"; git show "$tag":data/raw/bike.csv.dvc | grep -E "md5|size"
done
cat models/run.json

== dados-v1


- md5: fc7d423afbce5699a61ac1a8acc533b8
  size: 562742
  hash: md5
== dados-v2


- md5: 11e40c60d906dcbd0ebad321fcd221e0
  size: 850906
  hash: md5


{
  "run_id": "ef3dadd7da7444b5aeec434fa825ada7",
  "model_name": "bike-demand",
  "model_version": 

"7"
}

**Respostas esperadas:**
1. v1 `fc7d423a…` (562.742 bytes) e v2 `11e40c60…`. O `git show <tag>:<arquivo>` lê o ponteiro direto do histórico.
2. Porque o `.gitignore` ignora `data/**/*.csv`. O Git só enxerga o ponteiro `.dvc`; quem guarda o dado é o DVC.
3. O run mais recente tem `data_version = 11e40c60…` (dados v2), o mesmo md5 do `bike.csv.dvc`. Nos runs da T1 a tag era
   `nao-versionado`: ainda não existia o ponteiro `.dvc`. Com essa tag, o `run.json` e o Registry, dá para ir de uma
   versão do modelo até o dado exato que a treinou.